# CudaSIFT + RootSIFT (GPU) ORB-SLAM3 fork -- KITTI benchmark (Kaggle GPU)

Runs `orbslam3_sift_kitti_ate` (the SIFT-feature fork of ORB-SLAM3, now GPU-accelerated at the **extractor** via [Celebrandil/CudaSift](https://github.com/Celebrandil/CudaSift) + RootSIFT descriptor normalization) against a KITTI odometry sequence. This fork's earlier LightGlue **matcher** experiment (parts 56-58) was reverted after `[reconstruct-diag]` logging showed it producing ~95-97% geometrically-inconsistent matches on KITTI, and the LightGlue paper itself confirmed it was only ever trained/evaluated on wide-baseline data (MegaDepth tourist photos) -- see `DEBUGGING.md`. GPU-accelerating extraction instead sidesteps that negative result entirely.

**Before running:**
1. Notebook settings -> Accelerator -> GPU (T4 x2 or P100). Internet -> On.
2. Add Data -> attach a KITTI odometry (grayscale, sequence 00) dataset, or upload one -- needs `.../image_0/*.png` and a `poses/00.txt`-style ground-truth file somewhere under `/kaggle/input`.
3. Set `REPO_URL` below to this project's GitHub URL (must be pushed first -- this notebook only clones over HTTPS, so the repo needs to be public, or you'll need to bake a token into the URL yourself).

**IMPORTANT -- run `cudasift_probe` before trusting any tracking result**: CudaSift's keypoint scale/octave encoding is NOT the same as OpenCV's `cv::SIFT`, and `ORBextractor.cc`'s conversion is currently a best-effort placeholder pending real measurement (see `analyze/cudasift_probe.cpp`'s doc comment, and Session 14's Stage 0 precedent for why this matters). The setup cell below builds `cudasift_probe` automatically and prints the exact command to run it -- do that FIRST.

The heavy one-time setup (installing apt packages, building g2o from ORB-SLAM3's own `Thirdparty/g2o`, cloning CudaSift, downloading the ONNX Runtime GPU release -- still needed to compile `LightGlueMatcher.cc`, which stays in the tree unused) is cached under `kaggle/g2o_build`, `kaggle/cudasift_src`, and `kaggle/onnxruntime` for the rest of the session -- rerunning the run cell after tweaking `START_FRAME`/`MAX_FRAMES` does not repeat it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

In [ ]:
import os

REPO_URL = "https://github.com/nguyenhuunam852/ORB-SLAM-SIFT.git"  # set to your pushed repo
REPO_DIR = "/kaggle/working/ORB-SLAM-SIFT"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already present, skipping clone (rm -rf it to force a fresh clone)")

In [ ]:
# One-time setup: apt deps, g2o build, CudaSift clone, ONNX Runtime GPU
# download, cmake configure + build (USE_CUDASIFT=1 by default). Safe to
# rerun (skips steps whose output already exists). Also builds
# cudasift_probe and prints the command to run it -- do that before
# trusting the benchmark run below.
!cd {REPO_DIR} && bash kaggle/setup_and_run.sh

The cell above both builds AND runs a default 0-1000-frame benchmark (env var defaults in `kaggle/setup_and_run.sh`). To rerun with different frame ranges without repeating the build, use the cell below instead.

In [ ]:
import os

env = os.environ.copy()
env["SKIP_BUILD"] = "1"
env["USE_CUDASIFT"] = "1"  # set to "0" to compare against the plain CPU cv::SIFT path
env["START_FRAME"] = "0"
env["MAX_FRAMES"] = "1000"
env["OUT_PREFIX"] = "/kaggle/working/cudasift_1_1000"
# Uncomment and set explicitly if auto-detection under /kaggle/input picks the wrong dataset:
# env["KITTI_SEQ_DIR"] = "/kaggle/input/<your-dataset>/sequences/00"
# env["KITTI_POSES"] = "/kaggle/input/<your-dataset>/poses/00.txt"

import subprocess, sys
result = subprocess.run(["bash", "kaggle/setup_and_run.sh"], cwd=REPO_DIR, env=env)
sys.exit(result.returncode) if result.returncode != 0 else None

In [ ]:
# Fragment-by-fragment ATE report and the [sbp-diag] match-quality summary
# are printed directly to stdout by the run cell above (search for
# "[atlas-coverage]" / "[fragment" / "[sbp-diag]"). The keyframe trajectory
# itself is written to OUT_PREFIX + "_KeyFrameTrajectory.txt":
!tail -n 20 /kaggle/working/cudasift_1_1000_KeyFrameTrajectory.txt